In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
df_test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [5]:
df = df.pipe(lambda d: d.rename(columns=lambda c: str(c).lower().strip().replace(' ', '_')))
df.head()

,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
df_test = df_test.pipe(lambda d: d.rename(columns=lambda c: str(c).lower().strip().replace(' ', '_')))
df_test.head()

,passengerid,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [7]:
# EDA
target_col = 'survived'

if 'passengerid' in df.columns:
    df.drop(columns=['passengerid'], inplace=True)

if 'name' in df.columns:
    df.drop(columns=['name'], inplace=True)

if 'name' in df_test.columns:
    df_test.drop(columns=['name'], inplace=True)

df['sex'] = df['sex'].str.lower().map({'male':1,'female':0})
df_test['sex'] = df_test['sex'].str.lower().map({'male':1,'female':0})

df.head()

,survived,pclass,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,0,3,1,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,1,0,38.0,1,0,PC 17599,71.2833,C85,C
2,1,3,0,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,1,0,35.0,1,0,113803,53.1000,C123,S
4,0,3,1,35.0,0,0,373450,8.0500,NaN,S


In [8]:
df_test.head()

,passengerid,pclass,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,892,3,1,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,0,47.0,1,0,363272,7.0000,NaN,S
2,894,2,1,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,1,27.0,0,0,315154,8.6625,NaN,S
4,896,3,0,22.0,1,1,3101298,12.2875,NaN,S


In [9]:
# Missing age values
df['age'] = df['age'].fillna(df['age'].median())

In [10]:
df_test['age'] = df_test['age'].fillna(df_test['age'].median())

In [11]:
# Missing values in cabin column
print(df['cabin'].isna().sum() / len(df) * 100)

# Drop cabin column as it has more than 70% missing values
df.drop(columns=['cabin'], inplace=True)

77.10437710437711


In [12]:
# Missing values in cabin column
print(df_test['cabin'].isna().sum() / len(df_test) * 100)

df_test.drop(columns=['cabin'], inplace=True)

78.22966507177034


In [13]:
print(df['ticket'].nunique(), df['ticket'].isna().sum(), len(df))

# Ticket feature has high ratio (22%) of duplicate values (unique=681).
df.drop(columns=['ticket'], inplace=True)

681 0 891


In [14]:
print(df_test['ticket'].nunique(), df_test['ticket'].isna().sum(), len(df_test))
df_test.drop(columns=['ticket'], inplace=True)

363 0 418


In [15]:
df['fare'].fillna(df['fare'].mode()[0], inplace=True)
df_test['fare'].fillna(df_test['fare'].mode()[0], inplace=True)

C:\Users\SubudhiK\AppData\Local\Temp\ipykernel_17704\3513291345.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['fare'].fillna(df['fare'].mode()[0], inplace=True)
C:\Users\SubudhiK\AppData\Local\Temp\ipykernel_17704\3513291345.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a cop

In [16]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,1,22.0,1,0,7.2500,S
1,1,1,0,38.0,1,0,71.2833,C
2,1,3,0,26.0,0,0,7.9250,S
3,1,1,0,35.0,1,0,53.1000,S
4,0,3,1,35.0,0,0,8.0500,S


In [17]:
df_test.head()

,passengerid,pclass,sex,age,sibsp,parch,fare,embarked
0,892,3,1,34.5,0,0,7.8292,Q
1,893,3,0,47.0,1,0,7.0000,S
2,894,2,1,62.0,0,0,9.6875,Q
3,895,3,1,27.0,0,0,8.6625,S
4,896,3,0,22.0,1,1,12.2875,S


In [18]:
df['embarked'].isnull().sum()

np.int64(2)

In [19]:
df_test['embarked'].isnull().sum()

np.int64(0)

In [20]:
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
df_test['embarked'].fillna(df_test['embarked'].mode()[0], inplace=True)

C:\Users\SubudhiK\AppData\Local\Temp\ipykernel_17704\3228196785.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
C:\Users\SubudhiK\AppData\Local\Temp\ipykernel_17704\3228196785.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves 

In [21]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

df_encoded = encoder.fit_transform(df[['embarked']])

encoded_df = pd.DataFrame(
    df_encoded,
    columns=encoder.get_feature_names_out(['embarked']),
    index=df.index
)

df = pd.concat([df.drop(columns=['embarked']), encoded_df], axis=1)

df_test_encoded = encoder.transform(df_test[['embarked']])
encoded_test_df = pd.DataFrame(
    df_test_encoded,
    columns=encoder.get_feature_names_out(['embarked']),
    index=df_test.index
)
df_test = pd.concat([df_test.drop(columns=['embarked']), encoded_test_df], axis=1)

df.head(), df_test.head()


(   survived  pclass  sex   age  sibsp  parch     fare  embarked_C  embarked_Q  \
 0         0       3    1  22.0      1      0   7.2500         0.0         0.0   
 1         1       1    0  38.0      1      0  71.2833         1.0         0.0   
 2         1       3    0  26.0      0      0   7.9250         0.0         0.0   
 3         1       1    0  35.0      1      0  53.1000         0.0         0.0   
 4         0       3    1  35.0      0      0   8.0500         0.0         0.0   
 
    embarked_S  
 0         1.0  
 1         0.0  
 2         1.0  
 3         1.0  
 4         1.0  ,
    passengerid  pclass  sex   age  sibsp  parch     fare  embarked_C  \
 0          892       3    1  34.5      0      0   7.8292         0.0   
 1          893       3    0  47.0      1      0   7.0000         0.0   
 2          894       2    1  62.0      0      0   9.6875         0.0   
 3          895       3    1  27.0      0      0   8.6625         0.0   
 4          896       3    0  22.0    

In [22]:
scaler = StandardScaler()

scaled_values = scaler.fit_transform(
    df[['age', 'fare', 'pclass', 'sibsp', 'parch']]
)

scaled_df = pd.DataFrame(
    scaled_values,
    columns=['age', 'fare', 'pclass', 'sibsp', 'parch'],
    index=df.index
)

df[['age', 'fare', 'pclass', 'sibsp', 'parch']] = scaled_df


scaled_values = scaler.fit_transform(
    df_test[['age', 'fare', 'pclass', 'sibsp', 'parch']]
)

scaled_df_test = pd.DataFrame(
    scaled_values,
    columns=['age', 'fare', 'pclass', 'sibsp', 'parch'],
    index=df_test.index
)

df_test[['age', 'fare', 'pclass', 'sibsp', 'parch']] = scaled_df_test

df.head(), df_test.head()

(   survived    pclass  sex       age     sibsp     parch      fare  \
 0         0  0.827377    1 -0.565736  0.432793 -0.473674 -0.502445   
 1         1 -1.566107    0  0.663861  0.432793 -0.473674  0.786845   
 2         1  0.827377    0 -0.258337 -0.474545 -0.473674 -0.488854   
 3         1 -1.566107    0  0.433312  0.432793 -0.473674  0.420730   
 4         0  0.827377    1  0.433312 -0.474545 -0.473674 -0.486337   
 
    embarked_C  embarked_Q  embarked_S  
 0         0.0         0.0         1.0  
 1         1.0         0.0         0.0  
 2         0.0         0.0         1.0  
 3         0.0         0.0         1.0  
 4         0.0         0.0         1.0  ,
    passengerid    pclass  sex       age     sibsp     parch      fare  \
 0          892  0.873482    1  0.386231 -0.499470 -0.400248 -0.497063   
 1          893  0.873482    0  1.371370  0.616992 -0.400248 -0.511926   
 2          894 -0.315819    1  2.553537 -0.499470 -0.400248 -0.463754   
 3          895  0.873482    

In [23]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked_C,embarked_Q,embarked_S
0,0,0.827377,1,-0.565736,0.432793,-0.473674,-0.502445,0.0,0.0,1.0
1,1,-1.566107,0,0.663861,0.432793,-0.473674,0.786845,1.0,0.0,0.0
2,1,0.827377,0,-0.258337,-0.474545,-0.473674,-0.488854,0.0,0.0,1.0
3,1,-1.566107,0,0.433312,0.432793,-0.473674,0.420730,0.0,0.0,1.0
4,0,0.827377,1,0.433312,-0.474545,-0.473674,-0.486337,0.0,0.0,1.0


In [24]:

X_train = df.drop(columns=[target_col])
y_train = df[target_col]

X_test = df_test.drop(columns=['passengerid'])

In [25]:
# Logistic Regression model
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)
acc_logreg = round(logreg.score(X_train, y_train) * 100, 2)
acc_logreg

80.02

In [26]:
# Support Vector Machines
from sklearn.svm import SVC

svc = SVC(C=1.0, kernel='rbf', gamma='scale')
svc.fit(X_train, y_train)
y_pred = svc.predict(X_test)
acc_svc = round(svc.score(X_train, y_train) * 100, 2)
acc_svc

83.61

In [27]:
# knn model
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
acc_knn = round(knn.score(X_train, y_train) * 100, 2)
acc_knn

87.77

In [28]:
# Gaussian Naive Bayes
from sklearn.naive_bayes import GaussianNB

gaussian = GaussianNB()
gaussian.fit(X_train, y_train)
y_pred = gaussian.predict(X_test)
acc_gaussian = round(gaussian.score(X_train, y_train) * 100, 2)
acc_gaussian

78.56

In [29]:
# Perceptron
from sklearn.linear_model import Perceptron

perceptron = Perceptron(max_iter=1000, tol=1e-3, random_state=42)
perceptron.fit(X_train, y_train)
y_pred = perceptron.predict(X_test)
acc_perceptron = round(perceptron.score(X_train, y_train) * 100, 2)
acc_perceptron

72.5

In [30]:
# Linear SVC
from sklearn.svm import LinearSVC

linear_svc = LinearSVC(max_iter=1000, tol=1e-3, random_state=42)
linear_svc.fit(X_train, y_train)
y_pred = linear_svc.predict(X_test)
acc_linear_svc = round(linear_svc.score(X_train, y_train) * 100, 2)
acc_linear_svc

80.02

In [31]:
# Stochastic Gradient Descent
from sklearn.linear_model import SGDClassifier

sgd = SGDClassifier(max_iter=1000, tol=1e-3, random_state=42)
sgd.fit(X_train, y_train)
y_pred = sgd.predict(X_test)
acc_sgd = round(sgd.score(X_train, y_train) * 100, 2)
acc_sgd

77.55

In [32]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

decision_tree = DecisionTreeClassifier(random_state=42, ccp_alpha=0.01)
decision_tree.fit(X_train, y_train)
y_pred = decision_tree.predict(X_test)
acc_decision_tree = round(decision_tree.score(X_train, y_train) * 100, 2)
acc_decision_tree

81.93

In [33]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(n_estimators=100, random_state=42, min_samples_leaf=1, max_features=0.01)
random_forest.fit(X_train, y_train)
y_pred = random_forest.predict(X_test)
random_forest.score(X_train, y_train)
acc_random_forest = round(random_forest.score(X_train, y_train) * 100, 2)
acc_random_forest

97.98

In [34]:
models = pd.DataFrame({
    'Model': ['Support Vector Machines', 'KNN', 'Logistic Regression', 
              'Random Forest', 'Naive Bayes', 'Perceptron', 
              'Stochastic Gradient Decent', 'Linear SVC', 
              'Decision Tree'],
    'Score': [acc_svc, acc_knn, acc_logreg, 
              acc_random_forest, acc_gaussian, acc_perceptron, 
              acc_sgd, acc_linear_svc, acc_decision_tree]})
models.sort_values(by='Score', ascending=False)

,Model,Score
3,Random Forest,97.98
1,KNN,87.77
0,Support Vector Machines,83.61
8,Decision Tree,81.93
2,Logistic Regression,80.02
7,Linear SVC,80.02
4,Naive Bayes,78.56
6,Stochastic Gradient Decent,77.55
5,Perceptron,72.50


In [35]:
# ANN model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Initialize the ANN
ann = Sequential()
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10, restore_best_weights=True)

ann.add(Dense(units=32, activation='relu', input_shape=(X_train.shape[1],)))
ann.add(Dropout(0.2))
ann.add(Dense(units=16, activation='relu'))
ann.add(Dropout(0.2))
ann.add(Dense(units=8, activation='relu'))
ann.add(Dropout(0.3))
ann.add(Dense(units=1, activation='sigmoid'))
ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = ann.fit(X_train, y_train, batch_size=32, epochs=100, validation_split=0.2, callbacks=[es])
loss, acc_ann = ann.evaluate(X_train, y_train)
acc_ann = round(acc_ann * 100, 2)
acc_ann

Epoch 1/100


c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6124 - loss: 0.6695 - val_accuracy: 0.6704 - val_loss: 0.6380
Epoch 2/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6461 - loss: 0.6402 - val_accuracy: 0.7039 - val_loss: 0.5893
Epoch 3/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6348 - loss: 0.6182 - val_accuracy: 0.7486 - val_loss: 0.5501
Epoch 4/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6461 - loss: 0.6120 - val_accuracy: 0.7709 - val_loss: 0.5212
Epoch 5/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6868 - loss: 0.5929 - val_accuracy: 0.7765 - val_loss: 0.5063
Epoch 6/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6868 - loss: 0.5764 - val_accuracy: 0.7821 - val_loss: 0.4870
Epoch 7/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6980 - loss: 0.5781 - val_accuracy: 0.8045 - val_loss: 0.4676
Epoch 8/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7065 - loss: 0.5763 - val_accuracy: 0.8436 - val_loss

83.39

In [37]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

y_pred_prob = ann.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print(confusion_matrix(y_train, ann.predict(X_train) > 0.5))
print(classification_report(y_train, ann.predict(X_train) > 0.5))
print("ROC-AUC:", roc_auc_score(y_train, ann.predict(X_train)))

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
[[511  38]
 [110 232]]
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
              precision    recall  f1-score   support

           0       0.82      0.93      0.87       549
           1       0.86      0.68      0.76       342

    accuracy                           0.83       891
   macro avg       0.84      0.80      0.82       891
weighted avg       0.84      0.83      0.83       891

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
ROC-AUC: 0.8810223798719629


In [40]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scikeras.wrappers import KerasClassifier
import numpy as np

def create_model():
    ann = Sequential()
    ann.add(Dense(32, activation='relu', input_shape=(X_train.shape[1],)))
    ann.add(Dropout(0.2))
    ann.add(Dense(16, activation='relu'))
    ann.add(Dropout(0.2))
    ann.add(Dense(8, activation='relu'))
    ann.add(Dropout(0.3))
    ann.add(Dense(1, activation='sigmoid'))
    ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return ann

model = KerasClassifier(model=create_model, epochs=80, batch_size=32, verbose=0)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring='accuracy')

print("CV Accuracy scores:", scores)
print("Mean CV Accuracy:", scores.mean())
print("Std Dev:", scores.std())

c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the fi

c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


CV Accuracy scores: [0.83240223 0.81460674 0.78651685 0.83146067 0.83707865]
Mean CV Accuracy: 0.8204130311970372
Std Dev: 0.0185806251385802


In [41]:
submission = pd.DataFrame({
    'PassengerId': df_test['passengerid'],
    'Survived': y_pred.flatten()
})

# Save CSV
submission.to_csv('submission.csv', index=False)